### 调试数据集输出

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0' 

import warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*pretrained.*")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=UserWarning, message=".*Arguments other than a weight enum.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import torch
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
torch.backends.cudnn.deterministic = False  # 确保可以使用更多的 cuDNN 算法


torch.cuda.set_device(0)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
torch.backends.cudnn.deterministic = False  # 确保可以使用更多的 cuDNN 算法

import time
import argparse
import numpy as np
from sklearn.cluster import KMeans
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from mydatasets import adult, imdb, CIFAR10,news,covtype,MNIST,CIFAR100,Number,Mnist2MnistM
import math
from nets import *
import torch.nn as nn
import torch.optim as optim
from nets.nets_utils import EmbeddingRecorder

from utils.debug import *
from tqdm import tqdm
import random
from import_methods import *
from tqdm.contrib import tenumerate


def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果使用多个 GPU

start = time.time()
dataset_map = {
    'adult': adult,
    'imdb': imdb,
    'cifar10': CIFAR10,
    'mnist': MNIST,
    'news' : news,
    'covtype': covtype,
    'cifar100': CIFAR100,
    'number': Number,
    'mnist2mnistm': Mnist2MnistM,
}

args = argparse.Namespace(
    # data_path='/data2/wyh-dataset/image-dataset',
    # data_path='/data2/wyh-dataset/text-dataset/imdb',
    # data_path='/data2/wyh-dataset/text-dataset/News',
    data_path='/data2/wyh-dataset/tabular-dataset',
    dataset='adult',
    model='MLP',   
    method='LiSSA',
    selection_optimizer='SGD',
    selection_lr=0.01,
    selection_momentum=0.9,
    selection_weight_decay=5e-4,
    selection_nesterov=True,
    selection_batch=128,
    selection_fraction = 0.1,
    step_size=10,
    gamma=0.1,
    print_freq=10,
    gpu=[2],
    epochs=100,
    workers=4,
    random_seed=0,
    pretrain_epochs=3,
    active_fraction_ratio=0.5,
    Is_pretrain = True,
    proxy_model='ProxyMLP',
    proxy_epochs=3,
    dve_epochs=3,
    dve_lr=0.001,
    selection_interval = 10,
    lambda_reg = 0.5,
    balance = True,
    save_log_path = './log/',
    balance_target = 0,
    save_best_model_path = './best_model/',
    selection_framework = 'interval',
    patience = 100,
)

# 检查 dataset 是否在映射中
if args.dataset not in dataset_map:
    raise ValueError(f"未知的数据集：{args.dataset}")

seed = random.randint(0, 1000000)
# seed = 287854
print(f"随机种子：{seed}")
set_random_seed(seed)


print(f"CUDA_VISIBLE_DEVICES: {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"torch.cuda.current_device(): {torch.cuda.current_device()}")
print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")
print(f"torch.cuda.get_device_name(0): {torch.cuda.get_device_name(0)}")

# 获取数据集加载函数或类
dataset_func = dataset_map[args.dataset]
dataset_info = dataset_func(args.data_path)
channel, im_size, num_classes, class_names, mean, std, dst_train, dst_val, dst_test = dataset_info



In [ ]:

print(f"数据集信息：通道数={channel}，图像大小={im_size}，类别数={num_classes}")
print_input(args)

# 查看训练集的标签分布
print("train set label distribution:", Counter([int(dst_train[idx][1]) for idx in range(len(dst_train))])   )
print("val set label distribution:", Counter([int(dst_val[idx][1]) for idx in range(len(dst_val))])   )
print("test set label distribution:", Counter([int(dst_test[idx][1]) for idx in range(len(dst_test))])   )

run_method(args,channel,im_size,num_classes,class_names,mean,std,dst_train,dst_val,dst_test)


end = time.time()
print(f"总耗时：{end - start:.2f} 秒")
